# 04 - Workflow de Predição

Objetivo: carregar a pipeline salva, fazer predição batch e validar o contrato esperado pela API FastAPI.

Antes de rodar, treine o modelo com `python main.py` ou execute o notebook `03_training_evaluation.ipynb`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd

from src.api.schemas import PredictionRequest
from src.config.settings import Settings
from src.pipelines.prediction_pipeline import FraudPredictionService

pd.set_option("display.max_columns", 120)
settings = Settings(project_root=PROJECT_ROOT)
input_csv = settings.raw_data_dir / "transactions_data.csv"
settings.pipeline_path.exists(), settings.metadata_path.exists()

## 1. Carregar serviço de predição

In [ ]:
service = FraudPredictionService(settings)
service.threshold, service.metadata

## 2. Predição com registros manuais

O schema da API aceita registros flexíveis para suportar as colunas do dataset real. O importante é manter os mesmos nomes usados no treino sempre que possível.

In [ ]:
records = pd.read_csv(input_csv, nrows=1).to_dict(orient="records")

request = PredictionRequest(records=records)
service.predict_records(request.records)

## 3. Predição batch a partir de CSV

A saída inclui `fraud_score`, `is_fraud_predicted` e `threshold`.

In [ ]:
output_csv = PROJECT_ROOT / "artifacts" / "batch_predictions_sample.csv"

sample = pd.read_csv(input_csv, nrows=1000)
predictions = service.predict_frame(sample)
predictions.to_csv(output_csv, index=False)

predictions[["fraud_score", "is_fraud_predicted", "threshold"]].head(), output_csv

## 4. Inspeção dos maiores scores

Esta visão ajuda a entender quais transações o modelo considera mais arriscadas.

In [ ]:
predictions.sort_values("fraud_score", ascending=False).head(20)

## 5. Teste manual da API

Em um terminal na raiz do projeto:

```bash
uvicorn src.api.app:app --reload
```

Depois abra `http://127.0.0.1:8000/docs` para testar o endpoint `/predict`.